In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix


In [ ]:
# Read from CSV file
data = pd.read_csv("./Data/BreastCancerWisconsinDiagnosticDataSet.csv")

# Dataset information
dataset_name = "Breast Cancer Wisconsin"
dataset_source = "https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data?resource=download"  # e.g., "UCI ML Repository"
n_samples = data.shape[0]      # Total number of rows
n_features = data.shape[1]     # Number of features (excluding target)
problem_type = "binary_classification"  # "regression" or "binary_classification" or "multiclass_classification"

print(f"Dataset: {dataset_name}")
print(f"Source: {dataset_source}")
print(f"Samples: {n_samples}, Features: {n_features}")
print(f"Problem Type: {problem_type}")

np.random.seed(42)


In [ ]:
# Drop irrelevant columns and empty rows
# df = data.dropna() # drop records with missing values

df = data.dropna(how='all', axis=1)
print('\nMissing values in data:')
print(df.isnull().sum())

X = df.drop(columns=['id','diagnosis'] )
Y = (df['diagnosis']=='M').astype(int).values

print(f"df: {df}")
print(f"X: {X}")
print(f"Y: {Y}")


In [ ]:
# Divide the data in to training and test sets

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y) # to keep the class proportion similar in both test and train set


In [ ]:
# Model definitions
models = {"Logistic Regression": LogisticRegression(),
          "Decision Tree": DecisionTreeClassifier(),
          "KNN": KNeighborsClassifier(),
          "Naive Bayes": GaussianNB(),
          "Support Vector Machine": SVC(),
          }

# 2. Centralized parameter grid configurations
param_grids = {
    "Logistic Regression": {
        # Prefix with 'logisticregression__' because it's in a pipeline
        'logisticregression__max_iter': [100, 1000, 10000],
        'logisticregression__C': [0.1, 1.0, 10.0]
    },
    "Decision Tree": {
        # No prefix needed (not in a pipeline)
        'max_depth': [3, 5, 10, None],
        'criterion': ['gini', 'entropy']
    },
    "KNN": {
        # Prefix with 'kneighborsclassifier__' because it's in a pipeline
        'kneighborsclassifier__n_neighbors':[1,3,5],
        'kneighborsclassifier__weights': ['uniform', 'distance']
    },
    "Naive Bayes": {
        # No prefix needed (not in a pipeline)
        'var_smoothing': [1e-9, 1e-8, 1e-7]
    },
    "Support Vector Machine": {
        # Prefix with 'svc__' because it's in a pipeline
        'svc__kernel': ['rbf', 'linear'],
        'svc__C': [0.1, 1.0, 10.0],
        'svc__gamma': ['scale', 'auto']
    }
}



In [ ]:
results = [] # store the result
trained_models = {} # store the trained models

for name, model in models.items():
    # 3. Build the pipeline dynamically inside the loop
    if name in ["Decision Tree", "Naive Bayes"]:
        pipeline = model
    else:
        pipeline = make_pipeline(StandardScaler(), model)

    # 4. Set up Grid Search with the corresponding parameter grid
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    # 5. Run the search and fit the best parameters
    grid_search.fit(X_train, Y_train)

    # Extract the optimized model/pipeline
    best_model = grid_search.best_estimator_
    print(f"\nBest params for {name}: {grid_search.best_params_}")

    trained_models[name] = best_model
    # 6. Predict using the optimized model
    predictions = best_model.predict(X_test)

    if hasattr(best_model, "predict_proba"):
        Y_score = best_model.predict_proba(X_test)[:, 1]
    else:
        Y_score = best_model.decision_function(X_test)

    # Calculate metrics
    accuracy = accuracy_score(Y_test, predictions)
    auc = roc_auc_score(Y_test, Y_score)
    precision = precision_score(Y_test, predictions)
    recall = recall_score(Y_test, predictions)
    f1 = f1_score(Y_test, predictions)
    mcc = matthews_corrcoef(Y_test, predictions)

    results.append([name, accuracy, auc, precision, recall, f1, mcc])

    print("Confusion matrix:\n", confusion_matrix(Y_test, predictions))

In [ ]:
# Display the results in one table.
columns = ["ML Model Name", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
comparison = pd.DataFrame(results, columns=columns)
print(comparison.round(4).to_string(index=False))

# Select a winner using the highest F1 score.
best_row = comparison.loc[comparison["F1"].idxmax()]
print("Highest F1 model:", best_row["ML Model Name"])
print("F1 score:", round(best_row["F1"], 4))

In [ ]:

# Save the optimized pipeline
joblib.dump(trained_models, "./model/combined models.joblib")
print("\nSuccessfully saved all models into a single file: 'model/all_classifiers.joblib'")